In [ ]:
dataset_folder = '/mnt/Data1/Nick/transcription_pipeline/'
# dataset_folder = 'mnt/Data1/Hugo/transcription_pipeline/'
key = '001'

embryo_list = {
    '001': [
    'test_data/NSPARC/2025-08-01/MCP-mSG_His-RFP_RBS(001)(homo)_embryo01',
    'test_data/NSPARC/2025-08-01/MCP-mSG_His-RFP_RBS(001)(homo)_embryo02',
	],
	'003': [
	'test_data/NSPARC/2025-07-31/MCP-mSG_His-RFP_RBS(003)(homo)_embryo01',
	'test_data/NSPARC/2025-07-31/MCP-mSG_His-RFP_RBS(003)(homo)_embryo02',
	'test_data/NSPARC/2025-07-31/MCP-mSG_His-RFP_RBS(003)(homo)_embryo03',
	],
    '002': [ # Datasets with 1.5% laser power
        'test_data/NSPARC/2025-08-11/MCP-mSG_His-RFP_RBS(002)(het)_embryo01',
        'test_data/NSPARC/2025-08-11/MCP-mSG_His-RFP_RBS(002)(het)_embryo01_nc13',
        'test_data/NSPARC/2025-08-12/MCP-mSG_His-RFP_RBS(002)(het)_embryo01',
        'test_data/NSPARC/2025-08-12/MCP-mSG_His-RFP_RBS(002)(het)_embryo02',
    ]
}


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from transcription_pipeline import spot_pipeline
from transcription_pipeline.RateExtraction import FitAndAverage
from scipy.stats import linregress
from scipy.ndimage import gaussian_filter1d

# Parameters
nc14_start_frame = 0
min_frames = 40

results = []

def compute_binned_signal(signal, background, min_points=50, num_bins=10):
    # Auto-range for bins
    if len(background) == 0:
        return np.array([]), np.array([]), np.array([])

    min_bkg = np.percentile(background, 1)
    max_bkg = np.percentile(background, 99)
    bins = np.linspace(min_bkg, max_bkg, num_bins)
    bin_centers = (bins[:-1] + bins[1:]) / 2
    digitized = np.digitize(background, bins)

    mean_signal, std_signal, valid_centers = [], [], []

    for i in range(1, len(bins)):
        bin_points = signal[digitized == i]
        if len(bin_points) >= min_points:
            mean_signal.append(np.mean(bin_points))
            std_signal.append(np.std(bin_points) / np.sqrt(len(bin_points)))
            valid_centers.append(bin_centers[i - 1])

    return np.array(valid_centers), np.array(mean_signal), np.array(std_signal)

def plot_joint_signal_background(signal, background, output_path, min_points=20):
    g = sns.jointplot(x=background, y=signal, kind='hex', color='blue', alpha=0.8)
    centers, mean, err = compute_binned_signal(signal, background, min_points)

    if len(centers) > 0:
        g.ax_joint.plot(centers, mean, 'r-', linewidth=2, label='Mean Signal')
        g.ax_joint.fill_between(centers, mean - err, mean + err, color='red', alpha=0.2)
        g.ax_joint.legend()

    g.ax_joint.set_xlabel('Background')
    g.ax_joint.set_ylabel('Signal')
    g.fig.savefig(output_path)
    plt.show()
    plt.close(g.fig)

def plot_time_binned_signal(detected_spots, output_path, min_points=20):
    detected_spots = detected_spots.copy()
    detected_spots["t_min"] = detected_spots["t_s"] / 60
    detected_spots["t_bin"] = detected_spots["t_min"].astype(int)
    grouped = detected_spots.groupby("t_bin")

    num_plots = len(grouped)
    cols = 4
    rows = int(np.ceil(num_plots / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows))
    axes = axes.flatten()

    for i, (minute, group) in enumerate(grouped):
        signal = np.array(group["intensity_from_neighborhood"])
        background = np.array(group["background_intensity_from_neighborhood"])
        centers, mean, err = compute_binned_signal(signal, background, min_points)

        ax = axes[i]
        hb = ax.hexbin(background, signal, gridsize=30, cmap='Blues', bins='log')

        if len(centers) > 0:
            ax.plot(centers, mean, 'r-', linewidth=2)
            ax.fill_between(centers, mean - err, mean + err, color='red', alpha=0.2)

        # ax.set_xlim(np.percentile(background, 1), np.percentile(background, 99))
        # ax.set_ylim(np.percentile(signal, 1), np.percentile(signal, 99))
        ax.set_title(f'Time = {minute} min')
        ax.set_xlabel('Background')
        ax.set_ylabel('Signal')

    for j in range(i + 1, len(axes)):
        axes[j].axis('off')

    plt.tight_layout()
    plt.savefig(output_path)
    plt.show()
    plt.close(fig)

def plot_signal_and_background_over_time(detected_spots, output_path=None):
    """
    Plots mean signal and background intensity over time (1 point per minute),
    with SEM error bars. Does NOT plot signal-to-background ratio.
    """
    # Bin time to integer minutes
    detected_spots = detected_spots.copy()
    detected_spots["t_bin"] = (detected_spots["t_s"] / 60).astype(int)

    grouped = detected_spots.groupby("t_bin")

    # Compute mean and SEM
    avg_background = grouped["background_intensity_from_neighborhood"].mean()
    std_background = grouped["background_intensity_from_neighborhood"].std()

    avg_signal = grouped["intensity_from_neighborhood"].mean()
    std_signal = grouped["intensity_from_neighborhood"].std()

    count_per_bin = grouped.size()
    sem_background = std_background / np.sqrt(count_per_bin)
    sem_signal = std_signal / np.sqrt(count_per_bin)

    # Plotting
    fig, ax = plt.subplots(figsize=(10, 6))
    ax2 = ax.twinx()

    # Background (left axis)
    ax.errorbar(avg_background.index, avg_background, yerr=sem_background,
                fmt='o-', color='purple', ecolor='gray', capsize=3, label='Mean Background ± SEM')
    ax.set_ylabel('Background Intensity', color='purple')
    ax.tick_params(axis='y', labelcolor='purple')
    ax.grid(True)

    # Signal (right axis)
    ax2.errorbar(avg_signal.index, avg_signal, yerr=sem_signal,
                 fmt='s--', color='darkgreen', ecolor='lightgreen', capsize=3, label='Mean Signal ± SEM')
    ax2.set_ylabel('Signal Intensity', color='darkgreen')
    ax2.tick_params(axis='y', labelcolor='darkgreen')
    ax2.set_facecolor('none')

    # Combine legends
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

    ax.set_xlabel('Time (minutes)')
    ax.set_title('Mean Background and Signal Over Time')

    fig.tight_layout()
    if output_path:
        fig.savefig(output_path)

    plt.show()
    plt.close(fig)

import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d
import seaborn as sns
import pandas as pd

def smooth_time_residual_correlation(detected_spots, sigma=2, output_path=None, show_plot=True,
                                     show_time_trace=True, num_bins=30):
    """
    Analyze local correlation between signal and background intensity
    after removing developmental trends via Gaussian smoothing.

    Parameters
    ----------
    detected_spots : pd.DataFrame
        Must contain columns: 't_s', 'intensity_from_neighborhood', 'background_intensity_from_neighborhood'.
    sigma : float
        Standard deviation for Gaussian smoothing (in index space).
    output_path : str or None
        If provided, saves the residual jointplot to this path.
    show_plot : bool, default=True
        Whether to display the jointplot.
    show_time_trace : bool, default=False
        Whether to show signal and background with their smoothed trends.
    num_bins : int
        Number of bins for computing average residual curve.

    Returns
    -------
    corr : float
        Pearson correlation between signal and background residuals.
    """
    # Sort by time
    detected_spots = detected_spots.sort_values("t_s").reset_index(drop=True)

    signal = detected_spots["intensity_from_neighborhood"].values
    background = detected_spots["background_intensity_from_neighborhood"].values
    time = detected_spots["t_s"].values / 60  # Convert to minutes

    # Gaussian smoothing
    smoothed_signal = gaussian_filter1d(signal, sigma=sigma)
    smoothed_background = gaussian_filter1d(background, sigma=sigma)

    resid_signal = signal - smoothed_signal
    resid_background = background - smoothed_background

    # Pearson correlation
    corr = np.corrcoef(resid_signal, resid_background)[0, 1]
    print(f"Residual correlation (Gaussian detrended): {corr:.3f}")

    # Plot residuals using seaborn
    if show_plot or output_path:
        plot_df = pd.DataFrame({
            "Signal Residual": resid_signal,
            "Background Residual": resid_background
        })

        g = sns.jointplot(
            data=plot_df,
            x="Background Residual",
            y="Signal Residual",
            kind="hex",
            color="blue",
            # marginal_kws=dict(bins=40, fill=True),
            alpha=0.8,
        )

        # Add trend line: average signal residual in bins of background residual
        bin_edges = np.linspace(np.percentile(resid_background, 1),
                                np.percentile(resid_background, 99), num_bins + 1)
        bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
        digitized = np.digitize(resid_background, bin_edges)

        bin_means = []
        for i in range(1, len(bin_edges)):
            in_bin = resid_signal[digitized == i]
            bin_means.append(np.mean(in_bin) if len(in_bin) > 0 else np.nan)

        g.ax_joint.plot(bin_centers, bin_means, 'r-', label="Avg. signal residual")
        g.ax_joint.legend()

        g.fig.suptitle("Residuals After Gaussian Smoothing", y=1.02)
        g.ax_joint.set_xlabel("Background Residual")
        g.ax_joint.set_ylabel("Signal Residual")
        plt.show(g.fig)

        if output_path:
            g.savefig(output_path)
        if not show_plot:
            plt.close(g.fig)

    # Optional time trace
    if show_time_trace:
        fig, ax = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

        ax[0].plot(time, signal, alpha=0.4, label="Signal")
        ax[0].plot(time, smoothed_signal, 'r-', label="Smoothed Signal")
        ax[0].set_ylabel("Signal Intensity")
        ax[0].legend()

        ax[1].plot(time, background, alpha=0.4, label="Background")
        ax[1].plot(time, smoothed_background, 'g-', label="Smoothed Background")
        ax[1].set_xlabel("Time (min)")
        ax[1].set_ylabel("Background Intensity")
        ax[1].legend()

        plt.tight_layout()
        plt.show()

    return corr


In [ ]:
# --- Main Processing Loop ---
for group_name, embryo_list in embryo_list.items():
    print(f"######### Processing group: {group_name} #########")

    for embryo_id in embryo_list:
        test_dataset_name = os.path.join(dataset_folder, embryo_id)
        print(f"Dataset Path: {test_dataset_name}")

        # Load spot data
        spot_tracking = spot_pipeline.Spot()
        spot_tracking.read_results(name_folder=test_dataset_name)
        spot_df = spot_tracking.spot_dataframe
        detected_spots = spot_df[spot_df["particle"] != 0]

        signal = np.array(detected_spots['intensity_from_neighborhood'])
        background = np.array(detected_spots['background_intensity_from_neighborhood'])

        # Plot overall signal vs background
        output_path = os.path.join(test_dataset_name, 'signal_vs_background.png')
        plot_joint_signal_background(signal, background, output_path)

        # Plot time-binned signal vs background
        time_series_path = os.path.join(test_dataset_name, 'signal_vs_background_timeSeries.png')
        plot_time_binned_signal(detected_spots, time_series_path)

        # Plot signal and background as functions of time
        output_path = os.path.join(test_dataset_name, 'signal_vs_background_timeTraces.png')
        plot_signal_and_background_over_time(detected_spots, output_path)

        # Smooth time residual correlation
        output_path = os.path.join(test_dataset_name, 'signal_vs_background_residualCorr.png')
        smooth_time_residual_correlation(detected_spots, sigma=50, output_path=output_path)

        # # Uncomment if you want to include RNAP loading rate analysis
        # dataframe_path = os.path.join(test_dataset_name, 'compiled_dataframe.pkl')
        # if not os.path.exists(dataframe_path):
        #     print(f"Missing dataframe: {dataframe_path}, skipping.")
        #     continue
        #
        # compiled_dataframe = pd.read_pickle(dataframe_path)
        # fit_and_average = FitAndAverage(
        #     compiled_dataframe, nc14_start_frame, min_frames, num_bins, test_dataset_name
        # )
        # x, y, y_err, _, _ = fit_and_average.average_particle_fits()
        #
        # results.append({
        #     'group': group_name,
        #     'embryo_id': embryo_id,
        #     'avg_bkg': np.mean(background),
        #     'avg_signal': np.mean(signal),
        #     'max_signal': np.max(signal),
        #     'x': x,
        #     'y': y,
        #     'y_err': y_err
        # })

# average_fit_df = pd.DataFrame(results)



In [ ]:
spot_tracking = spot_pipeline.Spot()
spot_tracking.read_results(name_folder=test_dataset_name)
print(test_dataset_name)
spot_df = spot_tracking.spot_dataframe

In [ ]:
spot_dataframe_path =  os.path.join(test_dataset_name, 'spot_analysis_results/spot_dataframe.pkl')
test_load_spotDataframe = pd.read_pickle(spot_dataframe_path)

In [ ]:
spot_df.head()

In [ ]:
average_fit_df

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create side-by-side subplots
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# First subplot: avg_signal vs avg_bkg
sns.scatterplot(
    data=average_fit_df,
    x='avg_bkg',
    y='avg_signal',
    hue='group',
    s=100,
    ax=axes[0]
)
axes[0].set_xlabel('Average Background Intensity')
axes[0].set_ylabel('Average Signal Intensity')
axes[0].set_title('Avg Signal vs Background')
axes[0].grid(True)

# Second subplot: max_signal vs avg_bkg
sns.scatterplot(
    data=average_fit_df,
    x='avg_bkg',
    y='max_signal',
    hue='group',
    s=100,
    ax=axes[1],
    legend=False  # avoid duplicate legend
)
axes[1].set_xlabel('Average Background Intensity')
axes[1].set_ylabel('Max Signal Intensity')
axes[1].set_title('Max Signal vs Background')
axes[1].grid(True)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os

# Define the AP range of interest
ap_range = (0.15, 0.4)

# Create a dictionary to group embryos by group
grouped = average_fit_df.groupby('group')

group_averages = {}

for group_name, group_df in grouped:
    plt.figure(figsize=(10, 6))
    all_x, all_y, all_yerr = [], [], []

    for _, row in group_df.iterrows():
        x = np.array(row['x'])
        y = np.array(row['y'])
        y_err = np.array(row['y_err'])

        plt.errorbar(x, y, yerr=y_err, fmt='o', alpha=0.6, label=os.path.basename(row['embryo_id']))
        all_x.append(x)
        all_y.append(y)
        all_yerr.append(y_err)

    # Combine data by AP bin
    bin_dict = {}
    err_dict = {}

    for x_vals, y_vals, yerr_vals in zip(all_x, all_y, all_yerr):
        for xi, yi, yerri in zip(x_vals, y_vals, yerr_vals):
            xi_rounded = np.round(xi, 3)
            bin_dict.setdefault(xi_rounded, []).append(yi)
            err_dict.setdefault(xi_rounded, []).append(yerri)

    avg_bins = sorted(bin_dict.keys())

    # Mean and standard error propagation
    avg_rates = []
    avg_errors = []
    for b in avg_bins:
        vals = np.array(bin_dict[b])
        errs = np.array(err_dict[b])
        n = len(vals)
        mean = np.nanmean(vals)
        se = np.sqrt(np.nansum(errs ** 2)) / n  # Error propagation
        avg_rates.append(mean)
        avg_errors.append(se)

    group_averages[group_name] = (avg_bins, avg_rates, avg_errors)

    # Plot group average
    plt.plot(avg_bins, avg_rates, '-o', color='black', label=f'{group_name} Average')
    plt.fill_between(avg_bins,
                     np.array(avg_rates) - np.array(avg_errors),
                     np.array(avg_rates) + np.array(avg_errors),
                     color='black', alpha=0.2)

    # Highlight AP range of interest
    plt.axvspan(ap_range[0], ap_range[1], color='gray', alpha=0.15, label='AP range 0.15–0.4')

    # Compute and display mean over AP range
    ap_vals = np.array(avg_bins)
    rate_vals = np.array(avg_rates)
    range_mask = (ap_vals >= ap_range[0]) & (ap_vals <= ap_range[1])
    mean_in_range = np.nanmean(rate_vals[range_mask])
    print(f'{group_name}: Mean rate in AP range {ap_range} = {mean_in_range:.3f} AU')

    # Plot formatting
    plt.title(f'Slope Data: {group_name}')
    plt.xlabel('AP Position')
    plt.ylabel('Rate (AU/min)')
    plt.ylim(bottom=0)
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

# --- Combined group averages ---
plt.figure(figsize=(10, 6))

for group_name, (avg_bins, avg_rates, avg_errors) in group_averages.items():
    plt.plot(avg_bins, avg_rates, '-o', label=f'{group_name} Avg')
    plt.fill_between(avg_bins,
                     np.array(avg_rates) - np.array(avg_errors),
                     np.array(avg_rates) + np.array(avg_errors),
                     alpha=0.2)

# Highlight AP range
plt.axvspan(ap_range[0], ap_range[1], color='gray', alpha=0.15, label='AP range 0.15–0.4')

plt.title('Group Averages Overlay')
plt.xlabel('AP Position')
plt.ylabel('Rate (AU/min)')
plt.legend()
plt.ylim(bottom=0)
plt.grid(True)
plt.tight_layout()
plt.show()
